# GRF Training Notebook
**Google Research Football** — `academy_3_vs_1_with_keeper`

Separate notebook because GRF requires Python 3.10 (incompatible with 3.11/3.12).

## Workflow
1. Run all cells top-to-bottom to train and save results to Drive.
2. Results load automatically into `mappo_demo.ipynb`.

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================

GITHUB_URL    = 'https://github.com/keckster00/mappo'
DRIVE_RESULTS = '/content/drive/MyDrive/mappo_demo'  # must match main notebook

GRF_STEPS = 5_000_000
N_THREADS = 64  # GRF envs are heavier than MPE; 64 is safe on A100

print('Configuration loaded.')
print(f'  GRF_STEPS : {GRF_STEPS:,}')
print(f'  N_THREADS : {N_THREADS}')


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Drive mounted. Results folder: {DRIVE_RESULTS}')

REPO_PATH = '/content/mappo'
if not os.path.exists(REPO_PATH):
    print(f'Cloning repo from {GITHUB_URL}...')
    ret = os.system(f'git clone {GITHUB_URL} {REPO_PATH}')
    if ret != 0:
        raise RuntimeError('git clone failed. Check GITHUB_URL in the config cell.')
else:
    print(f'Repo already at {REPO_PATH}')

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')


---
## Setup — Python 3.10 + GRF
This takes **5-8 minutes**. Run once per Colab session.

In [ ]:
import subprocess, os, glob

def run(cmd, label):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0
    print(f'  [{"OK" if ok else "FAIL"}] {label}')
    if not ok:
        if result.stdout: print('  stdout:
' + result.stdout[-4000:])
        if result.stderr: print('  stderr:
' + result.stderr[-4000:])
    return ok

# ── Step 1: Python 3.10 ──────────────────────────────────────────────────
print('Step 1/7: Installing Python 3.10...')
run('add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1', 'deadsnakes PPA')
run('apt-get install -y python3.10 python3.10-dev python3.10-distutils > /dev/null 2>&1',
    'python3.10 + dev headers')
run('curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10 > /dev/null 2>&1',
    'pip for python3.10')

# ── Step 2: System libs ──────────────────────────────────────────────────
print('Step 2/7: Installing system libraries...')
run(
    'apt-get install -y libgl1-mesa-dev libsdl2-dev libsdl2-image-dev '
    'libsdl2-ttf-dev libsdl2-gfx-dev libboost-all-dev libboost-python3-dev '
    'libdirectfb-dev libst-dev mesa-utils xvfb x11vnc python3-tk cmake git '
    '> /dev/null 2>&1',
    'system libraries'
)

# ── Step 3: Clone with full submodules (no depth limit) ───────────────────
print('Step 3/7: Cloning gfootball + submodules (no depth limit, ~2 min)...')
run('rm -rf /content/football_src', 'clean old clone')
ok = run(
    'git clone --recurse-submodules '
    'https://github.com/google-research/football /content/football_src',
    'clone gfootball + submodules'
)
if not ok:
    raise RuntimeError('git clone failed.')

# Verify submodule cloned
engine_dir = '/content/football_src/third_party/gfootball_engine'
if not os.path.exists(engine_dir):
    print('  Submodule missing, running git submodule update...')
    run('cd /content/football_src && git submodule update --init --recursive',
        'submodule update')
print(f'  Engine dir exists: {os.path.exists(engine_dir)}')

# ── Step 4: Find and patch every CMakeLists.txt with Python 3.10 hint ────
print('Step 4/7: Patching CMakeLists.txt files to target Python 3.10...')
inject = (
    'set(Python_EXECUTABLE /usr/bin/python3.10)
'
    'set(Python3_EXECUTABLE /usr/bin/python3.10)
'
    'set(PYTHON_EXECUTABLE /usr/bin/python3.10)
'
    'set(Python_ROOT_DIR /usr)
'
)
cmake_files = glob.glob('/content/football_src/**/CMakeLists.txt', recursive=True)
print(f'  Found {len(cmake_files)} CMakeLists.txt file(s):')
for cmake_path in cmake_files:
    print(f'    {cmake_path}')
    with open(cmake_path, 'r') as f:
        content = f.read()
    if 'find_package(Python' in content or 'FIND_PACKAGE(Python' in content:
        if 'python3.10' not in content.lower():
            with open(cmake_path, 'w') as f:
                f.write(inject + content)
            print(f'    -> patched (contains find_package Python)')
        else:
            print(f'    -> already patched')
    else:
        print(f'    -> skipped (no Python find_package)')

# ── Step 5: Pre-install gym 0.21.0 separately ────────────────────────────
print('Step 5/7: Pre-installing gym 0.21.0...')
run("python3.10 -m pip install 'setuptools<60' wheel -q", 'setuptools<60')
ok = run(
    'python3.10 -m pip install --no-build-isolation gym==0.21.0 -q',
    'gym==0.21.0'
)
if not ok:
    raise RuntimeError('gym 0.21.0 install failed. See output above.')
run("python3.10 -m pip install 'setuptools>=65' -q", 'restore setuptools')

# ── Step 6: Build gfootball from patched source ───────────────────────────
print('Step 6/7: Building gfootball from patched source (~5 min)...')
grf_ok = run(
    'cd /content/football_src && '
    'python3.10 -m pip install --no-build-isolation --no-deps -v . 2>&1',
    'gfootball from patched source'
)
if not grf_ok:
    raise RuntimeError('GRF build failed. See full output above.')

# ── Step 7: ML deps ──────────────────────────────────────────────────────
print('Step 7/7: Installing PyTorch + ML deps under Python 3.10...')
run(
    'python3.10 -m pip install torch torchvision '
    '--index-url https://download.pytorch.org/whl/cu118 -q',
    'torch (cu118)'
)
run(
    'python3.10 -m pip install numpy==1.26.4 scipy seaborn imageio '
    'tensorboard tensorboardX setproctitle wandb -q',
    'ML deps'
)
run(f'python3.10 -m pip install -q -e {REPO_PATH}', 'onpolicy')

print('
All setup steps complete.')


In [ ]:
import subprocess

print('Verifying setup under Python 3.10...')
checks = [
    ("python3.10 -c 'import gfootball; print(\"gfootball ok\")'", 'gfootball import'),
    ("python3.10 -c 'import torch; print(torch.__version__, torch.cuda.is_available())'", 'torch + CUDA'),
    ("python3.10 -c 'import onpolicy; print(\"onpolicy ok\")'", 'onpolicy import'),
]
all_ok = True
for cmd, label in checks:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'FAIL'
    out = (r.stdout or r.stderr).strip()
    print(f'  [{status}] {label}: {out}')
    if r.returncode != 0:
        all_ok = False

if all_ok:
    print('\nAll checks passed. Safe to run training.')
else:
    raise RuntimeError('Checks failed. Fix errors above before training.')


---
## Training
Expect **30-60 min** on A100 for 5M steps. stdout streams live below.

In [ ]:
import subprocess, os, tempfile

os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
GRF_SCRIPT   = f'{REPO_PATH}/onpolicy/scripts/train/train_football.py'

GRF_ARGS = [
    '--env_name',           'Football',
    '--scenario_name',      'academy_3_vs_1_with_keeper',
    '--num_agents',         '3',
    '--algorithm_name',     'rmappo',
    '--experiment_name',    'demo',
    '--seed',               '1',
    '--n_training_threads', '1',
    '--n_rollout_threads',  str(N_THREADS),
    '--num_mini_batch',     '1',
    '--episode_length',     '200',
    '--num_env_steps',      str(GRF_STEPS),
    '--ppo_epoch',          '15',
    '--use_ReLU',
    '--gain',               '0.01',
    '--lr',                 '5e-4',
    '--critic_lr',          '5e-4',
    '--save_interval',      '25',
    '--log_interval',       '25',
    '--representation',     'simple115v2',
    '--rewards',            'scoring',
    '--share_reward',
    '--use_wandb',  # store_false: disables wandb, uses TensorBoard
]

env = os.environ.copy()
env['PYTHONPATH'] = REPO_PATH + ':' + env.get('PYTHONPATH', '')

# GRF requires a virtual display even in headless training
os.system('Xvfb :99 -screen 0 1280x720x24 &')
env['DISPLAY'] = ':99'

cmd = ['python3.10', GRF_SCRIPT] + GRF_ARGS
print(f'Starting GRF: {GRF_STEPS:,} steps, {N_THREADS} threads')
print('-' * 60)

with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as ef:
    err_path = ef.name

with open(err_path, 'w') as ef:
    result = subprocess.run(cmd, stderr=ef, text=True, env=env)

print('-' * 60)
if result.returncode == 0:
    print('Training complete!')
    os.unlink(err_path)
else:
    print(f'Training FAILED (exit {result.returncode})')
    print('--- last 80 lines of stderr ---')
    with open(err_path) as f:
        lines = f.readlines()
    print(''.join(lines[-80:]))
    print('--- end ---')
    os.unlink(err_path)


In [ ]:
import shutil, os

src = f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo'
dst = f'{DRIVE_RESULTS}/GRF_3v1'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Results saved to: {dst}')
    print('Load in mappo_demo.ipynb: set DEMO_MODE=True and run the GRF cell.')
else:
    print(f'No results found at {src}. Training may have failed.')


---
## Results — Training Curve

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dirs = sorted(glob.glob(
    f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo/run*/logs'))

if not log_dirs:
    print('No logs found. Run training first.')
else:
    ea = EventAccumulator(log_dirs[-1])
    ea.Reload()
    tags = ea.Tags().get('scalars', [])
    tag = 'average_episode_rewards' if 'average_episode_rewards' in tags else (tags[0] if tags else None)

    if tag:
        events = ea.Scalars(tag)
        steps  = [e.step  for e in events]
        values = [e.value for e in events]

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(steps, values, color='#4CAF50', linewidth=2)
        ax.set_xlabel('Environment Steps', fontsize=12)
        ax.set_ylabel('Average Episode Reward', fontsize=12)
        ax.set_title('MAPPO — GRF academy_3_vs_1_with_keeper', fontsize=14, fontweight='bold')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_path = f'{DRIVE_RESULTS}/tc2_grf_curve.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f'Final reward: {values[-1]:.4f}')
        print(f'Figure saved: {fig_path}')
        plt.show()
    else:
        print(f'No scalars found. Available tags: {tags}')
